In [1]:
import logging
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from spi_time_series import Pipeline, PipelineBuilder
from spi_time_series.data import Dataset
from spi_time_series.data.schemas import EventLog, PreprocessedData, RawData
from spi_time_series.evaluation import evaluate
from spi_time_series.features.extraction import extract_features_builder
from spi_time_series.features.log_based_features import BasicControlFlowFeatures
import pandas as pd

logging.basicConfig(level=logging.INFO)

## Build the pipeline

Use `PipelineBuilder` to compose a `Pipeline` from its strategy components.
Implement `my_preprocessor` and `my_splitter` below before calling `pipeline.run()`.

In [2]:
def my_preprocessor(raw: RawData) -> EventLog:
    """Clean the raw event log."""
    raise NotImplementedError


def my_splitter(log: EventLog) -> PreprocessedData:
    """Split the cleaned log into train/test sets."""
    raise NotImplementedError

def my_target_func(trace: pd.DataFrame, start_idx: int, end_idx: int) -> int | str:
    return end_idx - start_idx


pipeline = (
    PipelineBuilder()
    .with_dataset(Dataset())
    .with_preprocessor(my_preprocessor)
    .with_splitter(my_splitter)
    .with_feature_extractor(extract_features_builder([BasicControlFlowFeatures()], my_target_func))
    .add_model("logistic_regression", LogisticRegression()) # example of adding a model
    .add_model("decision_tree", DecisionTreeClassifier()) # example of adding a model
    .add_evaluator(evaluate)
    .build()
)

INFO:spi_time_series.data.dataset:Dataset found at C:\Users\roman\Documents\Uni\Aachen\Semester_3\SoftwarePraktikum\spi-time-series\data\raw\BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
c:\Users\roman\Documents\Uni\Aachen\Semester_3\SoftwarePraktikum\spi-time-series\.venv\Lib\site-packages\pm4py\utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.


## Run the full pipeline

In [3]:
evaluation = pipeline.run()

INFO:spi_time_series.pipeline.pipeline:Pipeline started.


NotImplementedError: 

## Run stages individually

Access each strategy through the `pipeline` instance to step through stages manually.
Useful for development and debugging.

In [ ]:
raw = RawData(event_log=pipeline.dataset.log)
raw.event_log.head()

,Selected,case:RequestedAmount,lifecycle:transition,Action,MonthlyCost,FirstWithdrawalAmount,EventOrigin,CreditScore,OfferID,Accepted,org:resource,case:concept:name,concept:name,OfferedAmount,case:ApplicationType,time:timestamp,EventID,NumberOfTerms,case:LoanGoal
0,None,20000.0,complete,Created,NaN,NaN,Application,NaN,None,None,User_1,Application_652823628,A_Create Application,NaN,New credit,2016-01-01 09:51:15.304000+00:00,Application_652823628,NaN,Existing loan takeover
1,None,20000.0,complete,statechange,NaN,NaN,Application,NaN,None,None,User_1,Application_652823628,A_Submitted,NaN,New credit,2016-01-01 09:51:15.352000+00:00,ApplState_1582051990,NaN,Existing loan takeover
2,None,20000.0,schedule,Created,NaN,NaN,Workflow,NaN,None,None,User_1,Application_652823628,W_Handle leads,NaN,New credit,2016-01-01 09:51:15.774000+00:00,Workitem_1298499574,NaN,Existing loan takeover
3,None,20000.0,withdraw,Deleted,NaN,NaN,Workflow,NaN,None,None,User_1,Application_652823628,W_Handle leads,NaN,New credit,2016-01-01 09:52:36.392000+00:00,Workitem_1673366067,NaN,Existing loan takeover
4,None,20000.0,schedule,Created,NaN,NaN,Workflow,NaN,None,None,User_1,Application_652823628,W_Complete application,NaN,New credit,2016-01-01 09:52:36.403000+00:00,Workitem_1493664571,NaN,Existing loan takeover


In [ ]:
cleaned = pipeline.preprocessor(raw)

NotImplementedError: 

In [ ]:
preprocessed = pipeline.splitter(cleaned)

NameError: name 'cleaned' is not defined

In [ ]:
features = pipeline.feature_extractor(preprocessed)

NameError: name 'preprocessed' is not defined

## Preprocessing
Preprocessing does 3 steps:
1) Data Cleaning (TBD)
2) Data Splitting into train and test (TBD)
3) Prefix Generation+ Target computation

### Data Cleaning
TODO

### Data Splitting
TODO

## Prefix generation
Prefix are created and returned as generators.

In [ ]:
from spi_time_series.preprocessing.preprocess import preprocess, sliding_window_factory

# define small dataframe for quick processing
dataset = Dataset()
small_cases = dataset.log["case:concept:name"].unique()[:5]
df_small = dataset.log[dataset.log["case:concept:name"].isin(small_cases)]

# raw = RawData(df_small)  # small datatset
raw = RawData(dataset.log)  # big dataset

INFO:spi_time_series.data.dataset:Dataset found at C:\Users\roman\Documents\Uni\Aachen\Semester_3\SoftwarePraktikum\spi-time-series\data\raw\BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.


In [5]:
prefix_generator = sliding_window_factory(min_length=3, max_length=None)

preprocessed_data = preprocess(
    raw, prefix_generator=prefix_generator
)

trace_sample = next(preprocessed_data.train_log)
start_idx, end_idx = next(trace_sample.prefix_indexes)

print(f"CASE ID: {trace_sample.case_id}")
print(pd.DataFrame(trace_sample.data[start_idx: end_idx], columns=list(preprocessed_data.col_idx.keys())))

CASE ID: Application_1691306052
  NumberOfTerms MonthlyCost FirstWithdrawalAmount OfferedAmount  \
0           NaN         NaN                   NaN           NaN   
1           NaN         NaN                   NaN           NaN   
2           NaN         NaN                   NaN           NaN   

  case:ApplicationType CreditScore       Action case:RequestedAmount  \
0           New credit         NaN      Created              10000.0   
1           New credit         NaN  statechange              10000.0   
2           New credit         NaN      Created              10000.0   

                  EventID lifecycle:transition          concept:name  \
0  Application_1691306052             complete  A_Create Application   
1     ApplState_284636842             complete           A_Submitted   
2      Workitem_831373279             schedule        W_Handle leads   

                    time:timestamp Accepted org:resource OfferID  \
0 2016-01-01 10:16:11.500000+00:00     None       Use

## Feature Extraction

In [7]:
from spi_time_series.features.extraction import extract_features_builder
from spi_time_series.features.log_based_features import BasicControlFlowFeatures


print(f"Number of train cases: {preprocessed_data.num_train_cases}")
feature_set = extract_features_builder([BasicControlFlowFeatures()], my_target_func)(preprocessed_data)

print(feature_set.X_train.head())

Number of train cases: 5


Processing cases: 100%|██████████| 5/5 [00:00<00:00, 126.96it/s]

   BasicControlFlowFeatures__elapsed_time_hours  \
0                                      0.000114   
1                                     20.502430   
2                                     20.519525   
3                                     20.519527   
4                                     20.519527   

   BasicControlFlowFeatures__prefix_length  \
0                                        3   
1                                        4   
2                                        5   
3                                        6   
4                                        7   

  BasicControlFlowFeatures__last_activity  \
0                          W_Handle leads   
1                          W_Handle leads   
2                          W_Handle leads   
3                  W_Complete application   
4                  W_Complete application   

   BasicControlFlowFeatures__count_W_Validate application  \
0                                                  0        
1                      